# 01 — vLLM offline (in-process) generation

The fastest way to generate from a model with **vLLM**: construct an `LLM` engine in-process
and batch-decode many prompts at once. This is the throughput path used for synthetic-data
generation.

**Library focus:** `vllm.LLM`, `vllm.SamplingParams`.

> First construction loads weights + captures CUDA graphs (tens of seconds). On this
> single-GPU box leave `tensor_parallel_size` at its default of 1.

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"

In [ ]:
from vllm import LLM, SamplingParams

# Single GPU -> no tensor_parallel_size. `dtype="auto"` honors the checkpoint dtype.
# For a longer context on a big model, add kv_cache_dtype="fp8".
llm = LLM(model=MODEL, dtype="auto", gpu_memory_utilization=0.9)

## Batched generation

`SamplingParams` controls decoding (temperature, top-p, max tokens, seed). `llm.generate`
takes a list of prompt strings and returns one `RequestOutput` per prompt; the text is at
`out.outputs[0].text`.

In [ ]:
prompts = [
    "The three laws of thermodynamics are",
    "A haiku about gradient descent:",
    "In one sentence, what is overfitting?",
]
sp = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=128, seed=0)

for out in llm.generate(prompts, sp):
    print("PROMPT:", out.prompt)
    print("OUTPUT:", out.outputs[0].text.strip())
    print("-" * 60)

## Chat-style generation

For an instruct model, `llm.chat(messages, sp)` applies the chat template for you. (The
manual equivalent is `tokenizer.apply_chat_template(...)` from notebook 00, then
`llm.generate` on the rendered string.)

In [ ]:
conversations = [
    [{"role": "user", "content": "Give me two uses for a paperclip."}],
    [{"role": "user", "content": "What is the boiling point of water at sea level?"}],
]
for out in llm.chat(
    conversations, SamplingParams(temperature=0.7, max_tokens=96, seed=0)
):
    print(out.outputs[0].text.strip())
    print("-" * 60)

> **Project pointer:** `llm_core.generation.generator` (`build_engine` / `generate`) wraps
> this with the repo's single-GPU defaults and the three temperature strategies; the CLI is
> `scripts/generate.py`. For an *already-running* server instead of in-process, see
> notebook 02.